## Setup: the P18 forecast joined with actuals, five segments (T0; C3 by format)
Segments S0–S4 as in the working list. `tablo` → `summary`: rows, forecast, actual, net bias %, WAPE %, share of empty-shelf rows. The table `a` (daily data with format, launch filter) feeds all later cells.

In [1]:
import pandas as pd
from pathlib import Path
import numpy as np

import os
D = Path(os.environ.get('DATA_DIR', 'data'))   # data folder (not included in the repository)

try:
    act = pd.read_parquet(D/'daily_store_item.parquet')
except Exception:
    act = pd.read_csv(D/'daily_store_item.csv.gz')
act['date'] = pd.to_datetime(act['date'])

fc     = pd.read_csv(D/'our_forecast_P18.csv', parse_dates=['date'])
items  = pd.read_csv(D/'items.csv')
stores = pd.read_csv(D/'stores.csv')
promo = pd.read_csv(D/'promo_calendar.csv', parse_dates=['start_date', 'end_date'])

p18_items = set(promo.loc[promo.promo_id == 'P18', 'item_id'])
p18_cats  = set(items.loc[items.item_id.isin(p18_items), 'category_id'])

# 1. attach actuals, category and format to every forecast row
df = (fc.merge(act, on=['date', 'store_id', 'item_id'], how='left')
        .merge(items, on='item_id')
        .merge(stores[['store_id', 'format']], on='store_id'))
assert df.units_sold.notna().all(), 'forecast rows without actuals'

# 2. label every row
is_p18   = df.item_id.isin(p18_items)
in_cat   = df.category_id.isin(p18_cats)
promo_wk = df.date.between('2025-06-16', '2025-06-22')
post_wk  = df.date.between('2025-06-23', '2025-07-06')
pre_wk   = df.date < '2025-06-16'

df['segment'] = 'S4 everything else'
df.loc[is_p18 & pre_wk,   'segment'] = 'S0 promoted items, 3 weeks before'
df.loc[is_p18 & promo_wk, 'segment'] = 'S1 promoted items, promotion week'
df.loc[is_p18 & post_wk,  'segment'] = 'S2 promoted items, next 2 weeks'
df.loc[~is_p18 & in_cat & promo_wk, 'segment'] = 'S3 same category, not promoted, promotion week'

# 3. aggregate by label
df['err']      = df.forecast - df.units_sold
df['censored'] = df.stock_on_hand_open <= df.units_sold

def summary(g):
    s = g.units_sold.sum()
    return pd.Series({
        'rows': len(g),
        'forecast': round(g.forecast.sum()),
        'actual': s,
        'net bias %': round(100 * g.err.sum() / s, 1),
        'WAPE %': round(100 * g.err.abs().sum() / s, 1),
        'empty-shelf rows %': round(100 * g.censored.mean(), 1),
    })

print(df.groupby('segment').apply(summary, include_groups=False))
print(df.groupby(['segment', 'format']).apply(summary, include_groups=False))
df['group'] = 'other categories'
df.loc[in_cat,  'group'] = 'same category, not promoted'
df.loc[is_p18,  'group'] = 'promoted item'
df['period'] = 'before'
df.loc[promo_wk, 'period'] = 'promotion week'
df.loc[post_wk,  'period'] = 'after'

print(df.groupby(['group', 'period']).apply(summary, include_groups=False))
print(df[df.segment.str.startswith('S4')].groupby('category_id').apply(summary, include_groups=False))

cat_of = items.set_index('item_id').category_id

a = act.merge(stores[['store_id', 'format']], on='store_id')
a['censored'] = a.stock_on_hand_open <= a.units_sold

# drop pre-launch days: for each store-item, nothing before the first positive stock date
first = a[a.stock_on_hand_open > 0].groupby(['store_id', 'item_id']).date.min().rename('first_stock')
a = a.join(first, on=['store_id', 'item_id'])
a = a[a.date >= a.first_stock].drop(columns='first_stock').reset_index(drop=True)
print('rows in a:', len(a), '| pre-launch rows removed:', len(act) - len(a))

                                                    rows  forecast    actual  \
segment                                                                        
S0 promoted items, 3 weeks before                12495.0   13554.0   13983.0   
S1 promoted items, promotion week                 4165.0   10894.0   10642.0   
S2 promoted items, next 2 weeks                   8330.0    9036.0    7857.0   
S3 same category, not promoted, promotion week    5124.0    8682.0    8350.0   
S4 everything else                              172494.0  264830.0  275771.0   

                                                net bias %  WAPE %  \
segment                                                              
S0 promoted items, 3 weeks before                     -3.1    79.4   
S1 promoted items, promotion week                      2.4    70.2   
S2 promoted items, next 2 weeks                       15.0    90.2   
S3 same category, not promoted, promotion week         4.0    71.3   
S4 everything else 

rows in a: 3471336 | pre-launch rows removed: 108072


## C1 — April 2025 in categories 1, 9, 11
Weekly sales 2025 vs the same weeks of 2024, then sales, stock and empty-shelf share by week.

In [2]:
act2 = act.merge(items, on='item_id')
act2['week'] = act2.date.dt.to_period('W-SUN').dt.start_time
wk = (act2[act2.category_id.isin([1, 9, 11])]
      .groupby(['category_id', 'week']).units_sold.sum().unstack(0))
print(wk.loc['2025-02-03':'2025-07-07'])
print(wk.loc['2024-02-05':'2024-07-08'])

category_id    1     9     11
week                         
2025-02-03   2525  2096  3524
2025-02-10   2570  2248  3580
2025-02-17   2603  2342  3600
2025-02-24   2583  2367  3557
2025-03-03   2571  3312  3630
2025-03-10   2625  2096  3479
2025-03-17   2662  2304  3603
2025-03-24   2503  2374  3711
2025-03-31   2549  2409  3487
2025-04-07   1692  1610  2302
2025-04-14   1498  1360  2027
2025-04-21   1413  1299  2004
2025-04-28   1431  1336  1998
2025-05-05   2351  2180  3321
2025-05-12   2577  2298  3525
2025-05-19   2607  2415  3703
2025-05-26   2605  2318  3308
2025-06-02   2559  2416  3562
2025-06-09   2573  2329  3617
2025-06-16   2997  2388  3615
2025-06-23   2444  2382  3500
2025-06-30   2453  2446  3457
2025-07-07   2607  2337  3657
category_id    1     9     11
week                         
2024-02-05   2633  2305  3557
2024-02-12   2548  3437  4475
2024-02-19   2647  2194  3165
2024-02-26   2509  2247  3495
2024-03-04   2535  2348  3537
2024-03-11   2558  2444  3586
2024-03-18

In [3]:
# Claim C1 — April 2025: weekly sales, stock and empty-shelf share, categories 1, 9, 11
act2['censored'] = act2.stock_on_hand_open <= act2.units_sold
m = act2.category_id.isin([1, 9, 11]) & act2.date.between('2025-02-03', '2025-07-13')
w = (act2[m].groupby(['category_id', 'week'])
       .agg(sales=('units_sold', 'sum'),
            stock=('stock_on_hand_open', 'sum'),
            empty_share=('censored', 'mean'))
       .round(2))
print(w.unstack(0))

            sales              stock               empty_share            
category_id    1     9     11     1      9      11          1     9     11
week                                                                      
2025-02-03   2525  2096  3524  14232  14070  19266        0.10  0.09  0.11
2025-02-10   2570  2248  3580  14140  13529  19162        0.12  0.12  0.11
2025-02-17   2603  2342  3600  13947  13463  19068        0.13  0.09  0.13
2025-02-24   2583  2367  3557  13976  13139  19238        0.12  0.13  0.13
2025-03-03   2571  3312  3630  13857  15408  19111        0.15  0.16  0.11
2025-03-10   2625  2096  3479  14111  13895  19224        0.12  0.10  0.12
2025-03-17   2662  2304  3603  13837  13339  19294        0.12  0.10  0.11
2025-03-24   2503  2374  3711  13899  13240  19048        0.13  0.12  0.10
2025-03-31   2549  2409  3487  14159  13266  19312        0.11  0.11  0.13
2025-04-07   1692  1610  2302   5672   5222   7161        0.37  0.39  0.40
2025-04-14   1498  1360  

## C2 — P18 promotion week by discount, category and format
Forecast vs sales for the P18 items grouped by discount; observed vs applied lift per item.

In [4]:
# Claim C2 — P18 promotion week by discount, category and format
p18 = promo[promo.promo_id == 'P18'][['item_id', 'discount']]
s1  = df[df.segment.str.startswith('S1')].merge(p18, on='item_id')

print(s1.groupby('discount').apply(summary, include_groups=False))
print(s1.groupby(['category_id', 'discount']).apply(summary, include_groups=False))
print(s1.groupby(['discount', 'format']).apply(summary, include_groups=False))

            rows  forecast  actual  net bias %  WAPE %  empty-shelf rows %
discount                                                                  
0.10       763.0    1971.0  1728.0        14.1    70.2                16.6
0.15      1841.0    5771.0  5557.0         3.9    66.3                21.0
0.40      1561.0    3152.0  3357.0        -6.1    76.7                31.2


                        rows  forecast  actual  net bias %  WAPE %  \
category_id discount                                                 
1           0.10       182.0     223.0   260.0       -14.2    71.1   
            0.15       392.0     645.0   736.0       -12.4    69.9   
3           0.10       392.0    1223.0   995.0        22.9    69.9   
            0.15       735.0    2693.0  2316.0        16.3    68.9   
7           0.10       189.0     525.0   473.0        11.1    70.4   
            0.15       714.0    2434.0  2505.0        -2.8    62.7   
13          0.40      1050.0    2333.0  2499.0        -6.6    79.4   
18          0.40       511.0     819.0   858.0        -4.6    68.6   

                      empty-shelf rows %  
category_id discount                      
1           0.10                    10.4  
            0.15                    11.0  
3           0.10                    18.4  
            0.15                    22.4  
7           0.10                    19.0 

In [5]:
# Claim C2 — P18 promotion week by discount, category and format

s0  = df[df.segment.str.startswith('S0')]
obs = pd.DataFrame({
    'pre_daily':     s0.groupby('item_id').units_sold.sum() / 21,
    'promo_daily': s1.groupby('item_id').units_sold.sum() / 7,
    'applied_lift':  s1.groupby('item_id').lift_applied.first(),
    'empty_share_S1':       s1.groupby('item_id').censored.mean().round(2),
}).join(p18.set_index('item_id')).join(items.set_index('item_id').category_id)
obs['observed_lift'] = (obs.promo_daily / obs.pre_daily).round(2)

print(obs.sort_values(['discount', 'category_id']))
print(obs.groupby('discount')[['observed_lift', 'applied_lift']].mean().round(2))

         pre_daily  promo_daily  applied_lift  empty_share_S1  discount  \
item_id                                                                   
1        19.190476    37.142857         2.199            0.10      0.10   
26       40.428571    84.857143         2.590            0.19      0.10   
30       27.714286    57.285714         2.590            0.18      0.10   
67       34.190476    67.571429         2.246            0.19      0.10   
4        31.571429    64.857143         2.199            0.12      0.15   
5        19.714286    40.285714         2.199            0.10      0.15   
21       15.523810    41.428571         2.590            0.20      0.15   
24       33.571429    75.857143         2.590            0.21      0.15   
27       25.904762    60.857143         2.590            0.21      0.15   
29       72.523810   152.714286         2.590            0.26      0.15   
62       16.333333    37.857143         2.246            0.20      0.15   
63       28.904762    61.

## Event table (P01–P18) and lift by discount — used by B1 and B3
One row per item-promotion: daily sales in the three weeks before and in the promotion week; lift = ratio.

In [6]:
# the table 'a' comes from the first cell (with the launch filter); it is not rebuilt here
rows, missing = [], []
for p in promo.itertuples():
    fmts = [f.strip() for f in p.formats.split(',')]
    m = (a.item_id == p.item_id) & a.format.isin(fmts)
    pre  = a[m & a.date.between(p.start_date - pd.Timedelta(days=21), p.start_date - pd.Timedelta(days=1))]
    week = a[m & a.date.between(p.start_date, p.end_date)]
    if len(week) == 0 or len(pre) == 0:
        missing.append((p.promo_id, p.item_id)); continue
    rows.append({
        'promo_id': p.promo_id, 'item_id': p.item_id, 'category_id': cat_of[p.item_id],
        'discount': p.discount, 'formats': p.formats,
        'pre_daily': pre.units_sold.sum() / 21,
        'promo_daily': week.units_sold.sum() / 7,
        'empty_share_pre': round(pre.censored.mean(), 2),
        'empty_share_promo': round(week.censored.mean(), 2),
    })

ev = pd.DataFrame(rows)
ev['lift'] = (ev.promo_daily / ev.pre_daily).round(2)
ev.to_csv('event_table.csv', index=False)
print('events:', len(ev), '| missing:', missing)

print(ev.groupby('discount').agg(n_events=('lift', 'size'), lift_mean=('lift', 'mean'),
                                 lift_median=('lift', 'median'), empty_share=('empty_share_promo', 'mean')).round(2))
print(ev.groupby(['category_id', 'discount']).lift.agg(['size', 'mean']).round(2).unstack(1))

events: 250 | missing: []
          n_events  lift_mean  lift_median  empty_share
discount                                               
0.10             4       2.02         2.02         0.16
0.15            26       2.28         2.30         0.22
0.20            40       2.34         2.35         0.26
0.25            74       2.36         2.37         0.26
0.30            53       2.32         2.30         0.23
0.35            42       2.25         2.25         0.21
0.40            11       2.45         2.49         0.28
            size                                   mean                    \
discount    0.10 0.15 0.20  0.25  0.30  0.35 0.40  0.10  0.15  0.20  0.25   
category_id                                                                 
1            1.0  2.0  NaN   2.0   2.0   7.0  NaN  1.94  2.04   NaN  2.16   
2            NaN  NaN  5.0   5.0   5.0   NaN  NaN   NaN   NaN  2.21  2.23   
3            2.0  9.0  5.0   6.0   NaN   NaN  NaN  2.08  2.35  2.43  2.69   
4       

## C3 — lift by format
The same events split by format: Hyper, Super, Express.

In [7]:
# Claim C3 — lift by format (item x promotion x format events)
rows = []
for p in promo.itertuples():
    for fmt in [f.strip() for f in p.formats.split(',')]:
        m = (a.item_id == p.item_id) & (a.format == fmt)
        pre  = a[m & a.date.between(p.start_date - pd.Timedelta(days=21), p.start_date - pd.Timedelta(days=1))]
        week = a[m & a.date.between(p.start_date, p.end_date)]
        if len(week) == 0 or pre.units_sold.sum() == 0:
            continue
        rows.append({
            'promo_id': p.promo_id, 'item_id': p.item_id, 'category_id': cat_of[p.item_id],
            'discount': p.discount, 'format': fmt,
            'pre_daily': pre.units_sold.sum() / 21,
            'promo_daily': week.units_sold.sum() / 7,
            'empty_share_promo': round(week.censored.mean(), 2),
        })

evf = pd.DataFrame(rows)
evf['lift'] = (evf.promo_daily / evf.pre_daily).round(2)
evf.to_csv('event_table_format.csv', index=False)

print(evf.groupby('format').agg(n_events=('lift', 'size'), lift_mean=('lift', 'mean'),
                                lift_median=('lift', 'median'), empty_share=('empty_share_promo', 'mean')).round(2))
print(evf.groupby(['category_id', 'format']).lift.mean().round(2).unstack(1))

         n_events  lift_mean  lift_median  empty_share
format                                                
Express       149       1.69         1.54         0.31
Hyper         250       2.54         2.54         0.29
Super         250       2.12         2.09         0.19
format       Express  Hyper  Super
category_id                       
1               1.61   2.55   1.97
2               1.42   2.44   2.13
3               1.87   2.67   2.23
4               1.95   2.55   2.25
5               1.80   2.35   1.98
6               1.48   2.65   2.28
7               1.99   2.48   2.09
8                NaN   2.32   1.81
9               1.45   2.62   2.24
10              2.07   2.47   2.00
11              1.40   2.53   2.02
12              1.44   2.35   1.87
13              1.72   2.83   2.42
14               NaN   2.54   2.23
15              1.79   2.72   2.28
16              1.36   2.49   2.01
18              1.55   2.29   1.92


## C5 — the two weeks after a promotion
P18 first, then every past promotion: sales in weeks +1 and +2 as a share of the pre-promotion level.

In [8]:
# Claim C5 — P18, the two weeks after the promotion
s2 = df[df.segment.str.startswith('S2')].copy()
s2['week'] = np.where(s2.date <= '2025-06-29', '+1', '+2')
print(s2.groupby('week').apply(summary, include_groups=False))
print(s2.groupby(['week', 'category_id']).apply(summary, include_groups=False))

        rows  forecast  actual  net bias %  WAPE %  empty-shelf rows %
week                                                                  
+1    4165.0    4518.0  3640.0        24.1    96.3                 8.6
+2    4165.0    4518.0  4217.0         7.1    85.0                 7.2
                    rows  forecast  actual  net bias %  WAPE %  \
week category_id                                                 
+1   1             574.0     395.0   425.0        -7.2    92.8   
     3            1127.0    1512.0  1236.0        22.3    92.4   
     7             903.0    1318.0  1117.0        18.0    81.8   
     13           1050.0     891.0   596.0        49.5   123.7   
     18            511.0     403.0   266.0        51.6   119.6   
+2   1             574.0     395.0   440.0       -10.3    89.2   
     3            1127.0    1512.0  1403.0         7.8    80.8   
     7             903.0    1318.0  1214.0         8.5    79.2   
     13           1050.0     891.0   815.0         9.3  

In [9]:
# Claim C5 — after P18 demand fell; check whether every past promotion shows the same dip.
# Cross-checked against the Power BI charts (56-day baseline).
rows = []
for p in promo.itertuples():
    fmts = [f.strip() for f in p.formats.split(',')]
    m = (a.item_id == p.item_id) & a.format.isin(fmts)
    pre = a[m & a.date.between(p.start_date - pd.Timedelta(days=21), p.start_date - pd.Timedelta(days=1))]
    w1  = a[m & a.date.between(p.end_date + pd.Timedelta(days=1), p.end_date + pd.Timedelta(days=7))]
    w2  = a[m & a.date.between(p.end_date + pd.Timedelta(days=8), p.end_date + pd.Timedelta(days=14))]
    base = pre.units_sold.sum() / 21
    if base == 0 or len(w2) == 0:
        continue
    rows.append({'promo_id': p.promo_id, 'item_id': p.item_id, 'category_id': cat_of[p.item_id],
                 'discount': p.discount,
                 'ratio_w1': round(w1.units_sold.sum() / 7 / base, 2),
                 'ratio_w2': round(w2.units_sold.sum() / 7 / base, 2)})
post = pd.DataFrame(rows)
print(post[['ratio_w1', 'ratio_w2']].describe().round(2))
print(post.groupby('discount')[['ratio_w1', 'ratio_w2']].mean().round(2))
print(post.groupby('category_id')[['ratio_w1', 'ratio_w2']].mean().round(2))

       ratio_w1  ratio_w2
count    250.00    250.00
mean       0.73      0.89
std        0.09      0.09
min        0.42      0.58
25%        0.66      0.84
50%        0.73      0.89
75%        0.78      0.94
max        0.96      1.17
          ratio_w1  ratio_w2
discount                    
0.10          0.88      0.92
0.15          0.81      0.92
0.20          0.74      0.90
0.25          0.73      0.88
0.30          0.71      0.87
0.35          0.69      0.90
0.40          0.66      0.83
             ratio_w1  ratio_w2
category_id                    
1                0.74      0.86
2                0.73      0.88
3                0.76      0.91
4                0.74      0.85
5                0.72      0.89
6                0.74      0.91
7                0.78      0.90
8                0.68      0.85
9                0.71      0.87
10               0.71      0.89
11               0.70      0.88
12               0.72      0.92
13               0.72      0.89
14               0.66    

## B4 — cannibalisation within the category
P18 segment S3 by category; then every past promotion: neighbours' sales vs their baseline, and the neighbours' loss as a share of the promoted items' extra sales.

In [10]:
# B4 — do promoted items take sales from the not-promoted items of their category? (P18, segment S3)
s3 = df[df.segment.str.startswith('S3')]
print(s3.groupby('category_id').apply(summary, include_groups=False))

               rows  forecast  actual  net bias %  WAPE %  empty-shelf rows %
category_id                                                                  
1            1288.0    1620.0  2001.0       -19.0    71.1                12.4
3             735.0     981.0   837.0        17.2    86.4                 6.3
7             973.0    1470.0  1481.0        -0.8    70.4                11.9
13            763.0    1844.0  1382.0        33.5    76.6                 7.6
18           1365.0    2767.0  2649.0         4.4    64.5                11.8


In [11]:
rows = []
for pid, g in promo.groupby('promo_id'):
    start, end = g.start_date.iloc[0], g.end_date.iloc[0]
    fmts = [f.strip() for f in g.formats.iloc[0].split(',')]
    promoted = set(g.item_id)
    for cat in cat_of[list(promoted)].unique():
        others = [i for i in cat_of[cat_of == cat].index if i not in promoted]
        m = a.item_id.isin(others) & a.format.isin(fmts)
        pre  = a[m & a.date.between(start - pd.Timedelta(days=21), start - pd.Timedelta(days=1))]
        week = a[m & a.date.between(start, end)]
        if pre.units_sold.sum() == 0: continue
        rows.append({'promo_id': pid, 'category_id': cat,
                     'n_promoted': len(promoted & set(cat_of[cat_of == cat].index)),
                     'n_neighbours': len(others),
                     'ratio': round(week.units_sold.sum() / 7 / (pre.units_sold.sum() / 21), 2)})
cann = pd.DataFrame(rows)
print(cann.groupby('category_id').ratio.agg(['size', 'mean', 'min', 'max']).round(2))
print(cann.groupby('n_promoted').ratio.agg(['size', 'mean']).round(2))

             size  mean   min   max
category_id                        
1               4  0.87  0.69  0.96
2               4  0.84  0.79  0.92
3               5  0.86  0.79  0.97
4               3  1.02  0.97  1.09
5               6  0.79  0.66  0.84
6               3  0.91  0.84  0.99
7               6  1.01  0.97  1.05
8               1  0.91  0.91  0.91
9               5  0.86  0.81  0.90
10              7  1.01  0.99  1.04
11              5  0.79  0.69  0.88
12              4  0.82  0.73  0.99
13              7  0.87  0.73  1.01
14              1  1.00  1.00  1.00
15              3  0.85  0.82  0.88
16              4  0.86  0.78  0.91
18              4  0.93  0.83  0.99
            size  mean
n_promoted            
2             15  0.99
3             20  0.89
4             27  0.87
5              8  0.81
6              2  0.79


In [12]:
"""Open question: do categories 4, 7 and 10 really show no loss, or were their promotions always small (few items)?"""
print(cann.groupby('category_id').n_promoted.agg(['mean', 'max']))

                 mean  max
category_id               
1            3.500000    5
2            3.750000    5
3            4.400000    6
4            3.333333    4
5            3.833333    5
6            3.000000    4
7            3.333333    5
8            4.000000    4
9            4.000000    4
10           2.714286    4
11           3.200000    4
12           3.250000    5
13           3.714286    6
14           2.000000    2
15           4.333333    5
16           3.250000    4
18           2.750000    3


In [13]:
"""
The reservation is partly right; separate the cases:

Cat. 7: 3.3 items on average, at most 5 — the same intensity as other categories, yet 1.01. No loss. A category property.
Cat. 4: 3.3 on average, at most 4 — categories 6 and 16 lose at similar intensity (0.91, 0.86); 4 does not. Probably real.
Cat. 10: 2.7 on average — low intensity; two-item promotions already give 0.99. "No loss" or "lightly loaded" cannot be separated.
Cat. 14: one promotion, 2 items — nothing to say.
"""

'\nThe reservation is partly right; separate the cases:\n\nCat. 7: 3.3 items on average, at most 5 — the same intensity as other categories, yet 1.01. No loss. A category property.\nCat. 4: 3.3 on average, at most 4 — categories 6 and 16 lose at similar intensity (0.91, 0.86); 4 does not. Probably real.\nCat. 10: 2.7 on average — low intensity; two-item promotions already give 0.99. "No loss" or "lightly loaded" cannot be separated.\nCat. 14: one promotion, 2 items — nothing to say.\n'

In [14]:
# B4 — neighbour loss from category totals.
# The ratio above does not show how much of the promoted items' extra sales came from neighbours,
# so compare the category's total: extra sales of promoted items vs the neighbours' loss.
rows = []
for pid, g in promo.groupby('promo_id'):
    start, end = g.start_date.iloc[0], g.end_date.iloc[0]
    fmts = [f.strip() for f in g.formats.iloc[0].split(',')]
    promoted = set(g.item_id)
    for cat in cat_of[list(promoted)].unique():
        in_cat_items = set(cat_of[cat_of == cat].index)
        prom = list(in_cat_items & promoted)
        others = list(in_cat_items - promoted)
        def week_and_base(item_list):
            m = a.item_id.isin(item_list) & a.format.isin(fmts)
            pre  = a[m & a.date.between(start - pd.Timedelta(days=21), start - pd.Timedelta(days=1))].units_sold.sum() / 3
            week = a[m & a.date.between(start, end)].units_sold.sum()
            return week, pre
        pw, pb = week_and_base(prom)
        ow, ob = week_and_base(others)
        rows.append({'promo_id': pid, 'category_id': cat, 'n_promoted': len(prom),
                     'extra_sales': pw - pb, 'neighbour_loss': ob - ow,
                     'category_net': (pw + ow) - (pb + ob)})
net = pd.DataFrame(rows)
net['loss_ratio'] = (net.neighbour_loss / net.extra_sales).round(2)
print(net[['extra_sales', 'neighbour_loss', 'category_net', 'loss_ratio']].describe().round(2))
print(net.groupby('n_promoted')[['extra_sales', 'neighbour_loss', 'loss_ratio']].mean().round(2))
print(net.groupby('category_id').loss_ratio.mean().round(2))

       extra_sales  neighbour_loss  category_net  loss_ratio
count        72.00           72.00         72.00       72.00
mean       1226.52          175.13       1051.39        0.14
std         550.48          154.33        478.15        0.13
min         273.67          -95.33        260.33       -0.26
25%         846.33           26.00        745.67        0.03
50%        1147.50          184.33        999.67        0.16
75%        1588.92          274.25       1463.92        0.21
max        2361.00          581.00       2090.00        0.45
            extra_sales  neighbour_loss  loss_ratio
n_promoted                                         
2                612.29           42.24        0.06
3               1086.33          198.57        0.18
4               1473.75          198.41        0.14
5               1779.33          248.04        0.14
6               1686.17          331.33        0.20
category_id
1     0.19
2     0.20
3     0.11
4    -0.03
5     0.19
6     0.09
7    -0.0

## C7 — empty-shelf days in the lift
Severity of empty-shelf days; the lift recomputed with those days excluded (before and promotion week).

In [15]:
# Claim C7 — shelves emptied in past promotion weeks, so the category lift learned from raw sales is too low.

# --- C7a: how severe are the empty-shelf days? ---
recent = a[a.date >= '2024-07-01'].copy()

pdays = pd.concat([pd.DataFrame({'date': pd.date_range(p.start_date, p.end_date), 'item_id': p.item_id})
                   for p in promo.itertuples()]).drop_duplicates()
recent = recent.merge(pdays.assign(promo=True), on=['date', 'item_id'], how='left')
recent['promo'] = recent.promo.fillna(False).astype(bool)
normal = recent[~recent.promo]

clean = normal[~normal.censored].groupby(['store_id', 'item_id']).units_sold.mean().rename('clean_mean')
normal = normal.join(clean, on=['store_id', 'item_id'])

cens = normal[normal.censored & (normal.clean_mean > 0)].copy()
cens['severity'] = pd.cut(cens.stock_on_hand_open / cens.clean_mean,
                        bins=[-1, 0, 0.5, 1, 99], labels=['zero stock', 'severe', 'moderate', 'mild'])

print('empty-shelf day share:', round(normal.censored.mean(), 3))
print(cens.severity.value_counts(normalize=True).round(2))
print(cens.groupby('format', observed=True).severity.value_counts(normalize=True).round(2).unstack())

# --- C7b: lift recomputed on days with a stocked shelf ---
rows = []
for p in promo.itertuples():
    for fmt in [f.strip() for f in p.formats.split(',')]:
        m = (a.item_id == p.item_id) & (a.format == fmt)
        pre  = a[m & a.date.between(p.start_date - pd.Timedelta(days=21), p.start_date - pd.Timedelta(days=1))]
        week = a[m & a.date.between(p.start_date, p.end_date)]
        pre_c, week_c = pre[~pre.censored], week[~week.censored]
        if len(pre) == 0 or len(week) == 0 or len(pre_c) == 0 or len(week_c) == 0 or pre_c.units_sold.mean() == 0:
            continue
        rows.append({'promo_id': p.promo_id, 'item_id': p.item_id, 'format': fmt, 'discount': p.discount,
                     'lift_raw':       week.units_sold.mean() / pre.units_sold.mean(),
                     'lift_uncensored': week_c.units_sold.mean() / pre_c.units_sold.mean()})
lc = pd.DataFrame(rows)
print('events:', len(lc))
print(lc.groupby('format')[['lift_raw', 'lift_uncensored']].mean().round(2))
print(lc.groupby('discount')[['lift_raw', 'lift_uncensored']].mean().round(2))

empty-shelf day share: 0.119
severity
zero stock    0.58
mild          0.34
moderate      0.06
severe        0.02
Name: proportion, dtype: float64
severity  zero stock  severe  moderate  mild
format                                      
Express         0.70    0.00      0.04  0.26
Hyper           0.36    0.04      0.10  0.49
Super           0.60    0.01      0.05  0.33


events: 647
         lift_raw  lift_uncensored
format                            
Express      1.70             1.99
Hyper        2.54             2.94
Super        2.12             2.25
          lift_raw  lift_uncensored
discount                           
0.10          1.84             1.92
0.15          2.18             2.43
0.20          2.18             2.40
0.25          2.21             2.54
0.30          2.18             2.41
0.35          2.17             2.40
0.40          2.33             2.85


## Ruled out — launch items

In [16]:
"""
Ruled-out explanation (ledger section 2):

Explanation: the six items launched on 3 March had low opening weeks that pulled the baseline down.
Test: weekly sales 3 Mar–7 Jul; these items' P18 errors.
Why ruled out: the ramp-up ended on 24 Mar, the window starts on 31 Mar; the six items' P18 errors are between −6 % and +4 %.
"""
new_items = [41, 57, 96, 136, 154, 167]
y = a[a.item_id.isin(new_items) & (a.date >= '2025-03-03')].copy()
y['week'] = y.date.dt.to_period('W-SUN').dt.start_time
print(y.groupby('week').units_sold.sum().loc[:'2025-07-07'])
print(df[df.item_id.isin(new_items)].groupby('item_id').apply(summary, include_groups=False))

week
2025-03-03     292
2025-03-10     726
2025-03-17    1171
2025-03-24    1603
2025-03-31    1675
2025-04-07    1699
2025-04-14    1996
2025-04-21    1724
2025-04-28    1713
2025-05-05    1688
2025-05-12    1732
2025-05-19    1769
2025-05-26    1736
2025-06-02    1735
2025-06-09    1683
2025-06-16    1685
2025-06-23    1695
2025-06-30    1708
2025-07-07    1715
Name: units_sold, dtype: int64
           rows  forecast  actual  net bias %  WAPE %  empty-shelf rows %
item_id                                                                  
41       1134.0    1089.0  1088.0         0.1    85.0                 9.6
57       1134.0    1274.0  1267.0         0.5    82.3                 9.5
96       1218.0    2338.0  2307.0         1.3    72.0                13.5
136      1092.0     920.0   979.0        -6.1    87.0                11.2
154      1218.0    1721.0  1661.0         3.6    81.1                10.5
167      1260.0    2984.0  2940.0         1.5    66.1                 9.8


## B2 — remodeled stores
Store-level lifts, 2025 promotions and 2023 promotions, by remodel status; Mann-Whitney within format.

In [17]:
# B2 — store-level lift per event, promotions from 2025 (after the remodel)
rows = []
for p in promo[promo.start_date >= '2025-01-01'].itertuples():
    fmts = [f.strip() for f in p.formats.split(',')]
    m = (a.item_id == p.item_id) & a.format.isin(fmts)
    pre  = a[m & a.date.between(p.start_date - pd.Timedelta(days=21), p.start_date - pd.Timedelta(days=1))]
    week = a[m & a.date.between(p.start_date, p.end_date)]
    g = (week.groupby('store_id').units_sold.sum() / 7) / (pre.groupby('store_id').units_sold.sum() / 21)
    for sid, l in g.dropna().items():
        if np.isfinite(l):
            rows.append({'promo_id': p.promo_id, 'item_id': p.item_id, 'store_id': sid, 'lift': l})
se = pd.DataFrame(rows).merge(stores[['store_id', 'format', 'remodeled_2024']], on='store_id')
print(stores.groupby(['format', 'remodeled_2024']).size())
print(se.groupby(['format', 'remodeled_2024']).lift.agg(['size', 'mean', 'median']).round(2))

from scipy import stats
for f, g in se.groupby('format'):
    x, y = g[g.remodeled_2024 == 1].lift, g[g.remodeled_2024 == 0].lift
    if len(x) and len(y):
        print(f, 'Mann-Whitney p =', round(stats.mannwhitneyu(x, y).pvalue, 3), '| diff =', round(x.mean() - y.mean(), 2))

format   remodeled_2024
Express  False             4
         True              4
Hyper    False             4
         True              4
Super    False             5
         True              9
dtype: int64
                        size  mean  median
format  remodeled_2024                    
Express False            110  1.69    1.29
        True             112  1.84    1.50
Hyper   False            251  2.50    2.40
        True             252  2.80    2.63
Super   False            314  2.22    2.00
        True             564  2.41    2.20


Express Mann-Whitney p = 0.125 | diff = 0.15
Hyper Mann-Whitney p = 0.0 | diff = 0.3
Super Mann-Whitney p = 0.016 | diff = 0.19


In [18]:
# B2 — stores remodeled in 2024 respond better to promotions?
"""
Same direction in all three formats, ~8-12 % higher lift; significant in two. That reads as "supports".

The caveat, and the question an interviewer will ask: this is a comparison, not an experiment.
Someone decided which stores to remodel - perhaps the already strong, large, central ones. Then the gap comes from
selection, not from the remodel. We cannot separate the two from this alone; the sentence is: "remodeled stores show
a higher lift; whether that is the remodel or the stores selected cannot be told from this data."

One extra test partly separates them: the same stores' lift before 2024. Higher already before the remodel -> selection;
equal before and higher after -> remodel. Five lines:
"""
rows = []
for p in promo[promo.start_date < '2024-01-01'].itertuples():
    fmts = [f.strip() for f in p.formats.split(',')]
    m = (a.item_id == p.item_id) & a.format.isin(fmts)
    pre  = a[m & a.date.between(p.start_date - pd.Timedelta(days=21), p.start_date - pd.Timedelta(days=1))]
    week = a[m & a.date.between(p.start_date, p.end_date)]
    g = (week.groupby('store_id').units_sold.sum() / 7) / (pre.groupby('store_id').units_sold.sum() / 21)
    rows += [{'store_id': s, 'lift': l} for s, l in g.dropna().items() if np.isfinite(l)]
before = pd.DataFrame(rows).merge(stores[['store_id', 'format', 'remodeled_2024']], on='store_id')
print(before.groupby(['format', 'remodeled_2024']).lift.agg(['size', 'mean']).round(2))


                        size  mean
format  remodeled_2024            
Express False             21  1.65
        True              25  1.52
Hyper   False            212  2.58
        True             212  2.76
Super   False            263  2.06
        True             476  2.34


## B1 — fast vs slow items
Speed quartiles: lift and extra units per day.

In [19]:
# B1 — "Fast-selling items respond less; promoting them is pointless"
# Source: the event table ev — 249 item-promotion events (P01–P18)

# Extra units per day brought by the promotion = promotion-week daily sales − normal daily sales
ev['extra_daily'] = ev.promo_daily - ev.pre_daily

# Split the events into 4 equal groups by normal sales rate (~62 events each)
ev['speed'] = pd.qcut(ev.pre_daily, 4, labels=['slow', 'mid-', 'mid+', 'fast'])

# Means per speed group
print(ev.groupby('speed', observed=True).agg(
    n_events      = ('lift', 'size'),             # events in the group
    speed_mean   = ('pre_daily', 'mean'),      # the group's mean normal daily sales
    lift      = ('lift', 'mean'),             # promotion week / normal  → the "responds less" part
    extra_units   = ('extra_daily', 'mean'),        # extra units per day      → the "not worth it" part
    empty_shelf   = ('empty_share_promo', 'mean'),  # empty-shelf share in the promotion week (high → lift suppressed)
).round(2))

       n_events  speed_mean  lift  extra_units  empty_shelf
speed                                                      
slow         63       11.78  2.40        16.22         0.23
mid-         62       23.63  2.33        31.36         0.24
mid+         62       41.26  2.31        54.60         0.23
fast         63       81.18  2.24        99.44         0.27


## C4 — P17's dip inside the P18 baseline window (checked, not claimed)

In [20]:
# C4 (not claimed) — P17's post-promotion dip (21 Apr–4 May) sits inside the P18 baseline window: is the P18 baseline of P17 items low?
p17_items = set(promo.loc[promo.promo_id == 'P17', 'item_id'])

t = df[df.segment.str.startswith(('S0', 'S4'))]          # non-promotion days (no lift)
t = t[~t.category_id.isin([1, 9, 11])]                 # categories with the April stock disruption excluded
t = t.assign(p17 = t.item_id.isin(p17_items))           # item was in P17?

print(t.groupby('p17').apply(summary, include_groups=False))
print(t.groupby(['p17', 'format']).apply(summary, include_groups=False))

           rows  forecast    actual  net bias %  WAPE %  empty-shelf rows %
p17                                                                        
False  145537.0  230745.0  231830.0        -0.5    70.8                11.6
True     8652.0   10399.0   10817.0        -3.9    76.1                11.9
                  rows  forecast    actual  net bias %  WAPE %  \
p17   format                                                     
False Express  26649.0   16000.0   15899.0         0.6   108.6   
      Hyper    43232.0  119087.0  119659.0        -0.5    60.6   
      Super    75656.0   95658.0   96272.0        -0.6    77.2   
True  Express   1260.0     600.0     624.0        -3.9   113.3   
      Hyper     2688.0    5642.0    5898.0        -4.3    63.5   
      Super     4704.0    4158.0    4295.0        -3.2    87.9   

               empty-shelf rows %  
p17   format                       
False Express                22.3  
      Hyper                   8.5  
      Super            

In [21]:
# C4 side check — did P17's dip spill into Express? (P17 did not run in Express)
p17_items = set(promo.loc[promo.promo_id == 'P17', 'item_id'])
e = a[a.item_id.isin(p17_items) & (a.format == 'Express') & a.date.between('2025-03-17', '2025-05-25')].copy()
e['week'] = e.date.dt.to_period('W-SUN').dt.start_time

print(e.groupby('week').agg(sales=('units_sold', 'sum'),
                             stock=('stock_on_hand_open', 'sum'),
                             empty_shelf=('censored', 'mean')).round(2))

            sales  stock  empty_shelf
week                                 
2025-03-17    162    928         0.17
2025-03-24    145    904         0.16
2025-03-31    153    925         0.17
2025-04-07    148    895         0.15
2025-04-14    162    923         0.18
2025-04-21    156    876         0.24
2025-04-28    146    924         0.19
2025-05-05    149    986         0.17
2025-05-12    149    900         0.20
2025-05-19    144    901         0.19


In [22]:
print(items[items.item_id.isin(p17_items)].groupby('category_id').item_id.apply(list))

category_id
5     [41, 44, 46, 48, 49]
7             [62, 67, 69]
13         [126, 127, 130]
Name: item_id, dtype: object


# Additional checks (added 24 Oct)

The cells below use the pipeline in `forecast_p19.py` (same folder; `data/` and `output/` next to it). They produce the evidence files cited in the ledger for T0, C1, C2, C3, C7, B1, B2, B3 and the ruled-out explanations. Run `python forecast_p19.py data output` once before this section (it writes `output/lift_records.csv`).

In [23]:
import sys, os, warnings
warnings.filterwarnings("ignore", category=FutureWarning)
sys.argv = ["forecast_p19.py", "data", "output"]   # data folder and output folder, next to this notebook
import forecast_p19 as F
import numpy as np, pandas as pd

## P18: company forecast vs this method vs actuals (T0, C1, C2, C3)
Five segments, then format, day and category breakdowns. Demand = sales on stocked-shelf days, filled on emptied days. Writes `output/p18_*.csv`.

In [24]:
"""Part 1 evidence: the company's P18 forecast (our_forecast_P18.csv) vs our method run at the same origin
(26 May 2025) vs actuals, by segment, format, day and category.
Run: python compare_p18.py <data_folder> <output_folder>"""
import numpy as np, pandas as pd

OUT = F.OUT
df, promo = F.df, F.promo
p18 = promo[promo["promo_id"] == "P18"]
ORIGIN, H0, H1 = pd.Timestamp("2025-05-26"), pd.Timestamp("2025-05-26"), pd.Timestamp("2025-07-06")
S, E = pd.Timestamp("2025-06-16"), pd.Timestamp("2025-06-22")
CAL = 1.074                                                   # calibration factor measured in the backtest

# our method at 26 May 2025 (only data before that date)
recs, post_recs, neigh_recs = F.promo_records(df, promo, drop_post=True)
g, choices, fac, _ = F.forecast(df, p18, ORIGIN, H0, H1, recs, post_recs, neigh_recs, True)
g.loc[g["plan_promo"].notna(), "forecast"] *= CAL
ours = g[["date", "store_id", "item_id", "forecast"]].rename(columns={"forecast": "ours"})

company = pd.read_csv(F.f("our_forecast_P18.csv"), parse_dates=["date"]).rename(columns={"forecast": "company"})

# actuals: sales; in the promotion week, empty-shelf days filled with the demand rule
act = df[(df["date"] >= H0) & (df["date"] <= H1)][["date", "store_id", "item_id", "format", "category_id",
                                                     "units_sold", "shelf_empty", "stock_on_hand_open", "promo_id"]].copy()
act["demand"] = act["units_sold"].astype(float)
m = act["promo_id"] == "P18"
act.loc[m, "demand"] = F.fill_demand(act[m])

x = (act.merge(company[["date", "store_id", "item_id", "company", "lift_applied"]], on=["date", "store_id", "item_id"])
        .merge(ours, on=["date", "store_id", "item_id"]))
p18_items = set(p18["item_id"]); p18_cats = set(F.cat_of[i] for i in p18_items)
def seg(r):
    if r["item_id"] in p18_items:
        if S <= r["date"] <= E: return "1 promoted item, promotion week"
        if E < r["date"] <= E + pd.Timedelta(days=14): return "2 promoted item, next 2 weeks"
        return "4 promoted item, other days"
    if r["category_id"] in p18_cats and S <= r["date"] <= E: return "3 same category, not promoted, promotion week"
    return "5 other items"
x["segment"] = x.apply(seg, axis=1)

def tbl(keys):
    t = x.groupby(keys).agg(sales=("units_sold", "sum"), demand=("demand", "sum"), company=("company", "sum"), ours=("ours", "sum"))
    for c in ["company", "ours"]:
        t[f"{c}/sales"] = t[c] / t["sales"]; t[f"{c}/demand"] = t[c] / t["demand"]
    return t.round(2)
t1 = tbl("segment"); print("By segment:\n", t1.to_string())
t2 = tbl(["segment", "format"]); print("\nBy segment x format:\n", t2.to_string())
w = x[x["segment"].str.startswith("1")]
t3 = w.groupby("date").agg(sales=("units_sold", "sum"), demand=("demand", "sum"), company=("company", "sum"), ours=("ours", "sum")).round(0)
print("\nPromotion week by day:\n", t3.to_string())
t4 = w.groupby("category_id").agg(sales=("units_sold", "sum"), demand=("demand", "sum"), company=("company", "sum"), ours=("ours", "sum"),
                                 company_lift=("lift_applied", "mean")).round(1)
t4["company/demand"] = (t4["company"] / t4["demand"]).round(2); t4["ours/demand"] = (t4["ours"] / t4["demand"]).round(2)
print("\nPromotion week by category:\n", t4.to_string())
oth = x[x["segment"] == "5 other items"]
t5 = oth.groupby("category_id").agg(sales=("units_sold", "sum"), company=("company", "sum"), ours=("ours", "sum"))
t5["company/sales"] = (t5["company"] / t5["sales"]).round(3); t5["ours/sales"] = (t5["ours"] / t5["sales"]).round(3)
print("\nOther items (baseline check) by category, forecast/sales:\n", t5[["sales", "company/sales", "ours/sales"]].to_string())
x["ae_c"] = (x["company"] - x["units_sold"]).abs(); x["ae_o"] = (x["ours"] - x["units_sold"]).abs()
print("\nAbsolute error vs sales, store-item-day: company", round(x["ae_c"].sum()), " ours", round(x["ae_o"].sum()))
for t, n in [(t1, "p18_segments.csv"), (t2, "p18_segments_format.csv"), (t3, "p18_days.csv"), (t4, "p18_categories.csv"), (t5, "p18_baseline_by_category.csv")]:
    t.to_csv(os.path.join(OUT, n))


By segment:
                                                 sales     demand    company       ours  company/sales  company/demand  ours/sales  ours/demand
segment                                                                                                                                       
1 promoted item, promotion week                 10642   13297.92   10894.21   13417.09           1.02            0.82        1.26         1.01
2 promoted item, next 2 weeks                    7857    7857.00    9036.12    7347.32           1.15            1.15        0.94         0.94
3 same category, not promoted, promotion week    8350    8350.00    8682.14    8338.38           1.04            1.04        1.00         1.00
4 promoted item, other days                     13983   13983.00   13554.18   13706.86           0.97            0.97        0.98         0.98
5 other items                                  275771  275771.00  264829.88  273909.24           0.96            0.96        0.99

## Discount depth, within item (B3)
A: across items (flat). B: pairs of the same item and format at different discounts. Writes `output/discount_pairs.csv`.

In [25]:
"""
Part 2, belief 3: "Discount depth barely matters. A 20% discount sells about as well as a 40% one."

Two tests on the same records (output/lift_records.csv, produced by forecast_p19.py; unit: item x promotion x format;
num = promotion-week sales on full-shelf store-days, den = baseline on the same store-days, lift = num / den):

  A. Across items: group promotions by discount, average the lifts. Cannot separate the discount from which items
     received deep discounts.
  B. Within item: same item, same format, two promotions at different discounts = one pair. Compare the pair's
     lifts. The item is held fixed, so "which item got the deep discount" no longer matters.

Run: python test_discount_depth.py output/lift_records.csv
"""
import itertools
import numpy as np, pandas as pd
from math import comb

path = "output/lift_records.csv"
r = pd.read_csv(path)
r = r[(r["num"] > 0) & (r["den"] > 0)].copy()
r["L"] = r["num"] / r["den"]
print(f"Records: {len(r)} item x promotion x format, {r['item_id'].nunique()} items, {r['promo_id'].nunique()} promotions")

# ---------------------------------------------------------------- A. across items
a = r.groupby("discount").agg(n=("L", "size"), mean_lift=("L", "mean"), num=("num", "sum"), den=("den", "sum"))
a["sum_over_sum_lift"] = a["num"] / a["den"]
a["promotions"] = r.groupby("discount")["promo_id"].agg(lambda s: ",".join(sorted(set(s))))
print("\nA. Across items, by discount:")
print(a[["n", "mean_lift", "sum_over_sum_lift", "promotions"]].round(2).to_string())
print("   -> flat between 15% and 35%; the 10% row is P18 only, the 40% row is P03 (2 items) and P18.")

# ---------------------------------------------------------------- B. within-item pairs
rows = []
for (it, fm), g in r.groupby(["item_id", "format"]):
    for x, y in itertools.combinations(g.itertuples(), 2):
        if x.discount == y.discount:
            continue
        hi, lo = (x, y) if x.discount > y.discount else (y, x)
        rows.append(dict(item_id=it, format=fm, shallow=lo.discount, deep=hi.discount, gap=hi.discount - lo.discount,
                         log_ratio=np.log(hi.L / lo.L), p_shallow=lo.promo_id, p_deep=hi.promo_id))
p = pd.DataFrame(rows)
n_pairs, n_items = len(p), p["item_id"].nunique()
ratio = float(np.exp(p["log_ratio"].mean()))
slope10 = float(np.exp(p["log_ratio"].sum() / p["gap"].sum() * 0.10))
share_pos = float((p["log_ratio"] > 0).mean())
print(f"\nB. Within item: {n_pairs} pairs, {n_items} items")
print(f"   typical deep/shallow lift ratio: {ratio:.3f}; a 10-point deeper discount multiplies the lift by {slope10:.3f}; "
      f"deep > shallow in {share_pos:.0%} of pairs")
p["gap_group"] = pd.cut(p["gap"], [0, 0.05, 0.10, 0.15, 0.30], labels=["<=5 pts", "6-10", "11-15", "16-30"])
print(p.groupby("gap_group", observed=True).agg(n=("log_ratio", "size"),
      deep_over_shallow=("log_ratio", lambda s: round(float(np.exp(s.mean())), 3))).to_string())
k, n = int((p["log_ratio"] > 0).sum()), n_pairs
p_value = sum(comb(n, i) for i in range(k, n + 1)) / 2 ** n
print(f"   sign test: deep > shallow in {k}/{n} pairs, H0 = 50%, one-sided p = {p_value:.2e}")
q = p[(p["p_deep"] != "P18") & (p["p_shallow"] != "P18")]
print(f"   excluding P18: {len(q)} pairs, ratio {np.exp(q['log_ratio'].mean()):.3f}, per 10 pts x{np.exp(q['log_ratio'].sum()/q['gap'].sum()*0.10):.3f}")
print("   per-10-pt effect by format:", {f: round(float(np.exp(g['log_ratio'].sum() / g['gap'].sum() * 0.10)), 3) for f, g in p.groupby("format")})
p.to_csv(os.path.join(os.path.dirname(path) or ".", "discount_pairs.csv"), index=False)


Records: 646 item x promotion x format, 133 items, 18 promotions

A. Across items, by discount:
            n  mean_lift  sum_over_sum_lift                                                           promotions
discount                                                                                                        
0.10       12       1.94               2.16                                                                  P18
0.15       73       2.29               2.43                                  P01,P05,P08,P10,P14,P15,P16,P17,P18
0.20      106       2.43               2.51          P01,P02,P03,P04,P05,P06,P07,P08,P09,P10,P12,P14,P15,P16,P17
0.25      192       2.54               2.58  P01,P02,P03,P04,P05,P06,P07,P08,P09,P10,P11,P12,P13,P14,P15,P16,P17
0.30      129       2.41               2.45  P01,P02,P03,P04,P05,P06,P07,P08,P09,P10,P11,P12,P13,P14,P15,P16,P17
0.35      106       2.37               2.47              P02,P03,P04,P05,P06,P07,P09,P10,P11,P12,P13,P14,P15,P17



B. Within item: 293 pairs, 55 items
   typical deep/shallow lift ratio: 1.092; a 10-point deeper discount multiplies the lift by 1.113; deep > shallow in 66% of pairs
             n  deep_over_shallow
gap_group                        
<=5 pts    131              1.036
6-10       118              1.115
11-15       10              1.285
16-30       34              1.181
   sign test: deep > shallow in 193/293 pairs, H0 = 50%, one-sided p = 3.00e-08
   excluding P18: 212 pairs, ratio 1.053, per 10 pts x1.079
   per-10-pt effect by format: {'Express': 1.104, 'Hyper': 1.134, 'Super': 1.096}


## Fast vs slow items (B1) and remodeled stores (B2)
B1: speed quartiles within format, lift and extra units. B2: store × promotion lifts by remodel status in 2023 and 2025, Mann-Whitney within format, and the same stores' 2023→2025 change (difference in differences).

In [26]:
"""
Part 2, belief 1: "Fast-selling items respond less to promotions than slow-selling items, so there is little
                   point promoting them."
Part 2, belief 2: "Stores remodeled in 2024 respond better to promotions than the other stores."

Data: output/lift_records.csv (item x promotion x format) and the daily data for store-level lifts.
Run: python test_beliefs_1_2.py <data_folder> <output_folder>  (same arguments as forecast_p19.py)
"""
import numpy as np, pandas as pd
from scipy import stats

OUT = F.OUT
df, promo, stores = F.df, F.promo, F.stores

# ============================================================ BELIEF 1: fast vs slow items
r = pd.read_csv(os.path.join(OUT, "lift_records.csv"))
r = r[(r["num"] > 0) & (r["den"] > 0)].copy()
r["L"] = r["num"] / r["den"]
r["base_daily"] = r["den"] / r["n_days"]              # the item's daily baseline per store-day in that format = speed
r["speed"] = r.groupby("format")["base_daily"].transform(lambda s: pd.qcut(s, 4, labels=["Q1 slow", "Q2", "Q3", "Q4 fast"]))
r["extra_units"] = (r["num"] - r["den"]) / r["n_days"]  # extra units per store-day brought by the promotion
t = r.groupby(["format", "speed"], observed=True).agg(n=("L", "size"), base=("base_daily", "mean"),
                                                     mean_lift=("L", "mean"), median_lift=("L", "median"),
                                                     extra_units=("extra_units", "mean"))
print("Belief 1 - lift and extra units per store-day, by speed quartile within format:")
print(t.round(2).to_string())
print("\nSpearman correlation log(baseline) ~ log(lift), by format:")
for fm, g in r.groupby("format"):
    rho, p = stats.spearmanr(np.log(g["base_daily"]), np.log(g["L"]))
    print(f"  {fm:8s} n={len(g):3d}  rho={rho:+.3f}  p={p:.3g}")
t.round(3).to_csv(os.path.join(OUT, "belief1_speed_quartiles.csv"))

# ============================================================ BELIEF 2: remodeled stores
# Store x promotion lift (sum / sum over full-shelf store-days, baseline rebuilt at the promotion's start).
# No remodel dates are given, so: 2023 promotions = before, 2025 promotions = after, 2024 = transition.
rows = []
for pid, grp in promo.groupby("promo_id"):
    start, end = grp["start_date"].iloc[0], grp["end_date"].iloc[0]
    b, prof = F.baseline(df, start, drop_post=True)
    wk = df[(df["date"] >= start) & (df["date"] <= end) & ~df["shelf_empty"] & (df["promo_id"] == pid)]
    wk = F.daily_base(wk, b, prof)
    for sid, g in wk.groupby("store_id"):
        rows.append(dict(promo_id=pid, store_id=sid, num=g["units_sold"].sum(), den=g["base_day"].sum(), n_days=len(g)))
sl = pd.DataFrame(rows).merge(stores, on="store_id")
sl = sl[sl["den"] > 0]; sl["L"] = sl["num"] / sl["den"]
sl["period"] = np.where(sl["promo_id"] <= "P04", "2023 (before)", np.where(sl["promo_id"] >= "P15", "2025 (after)", "2024 (transition)"))
sl.to_csv(os.path.join(OUT, "belief2_store_lifts.csv"), index=False)

print("\nBelief 2 - store x promotion lift by remodel status (sum / sum, within format):")
tab = sl.groupby(["period", "format", "remodeled_2024"]).agg(n_store_promos=("L", "size"), num=("num", "sum"), den=("den", "sum"))
tab["lift"] = tab["num"] / tab["den"]; tab = tab.drop(columns=["num", "den"])
print(tab.round(2).to_string())
print("\n2025 promotions (after the remodel), within format, Mann-Whitney on store x promotion lifts:")
for fm, g in sl[sl["period"] == "2025 (after)"].groupby("format"):
    a, bb = g.loc[g["remodeled_2024"], "L"], g.loc[~g["remodeled_2024"], "L"]
    u, p = stats.mannwhitneyu(a, bb, alternative="two-sided")
    print(f"  {fm:8s} remodeled n={len(a):2d} median {a.median():.2f} | others n={len(bb):2d} median {bb.median():.2f} | p={p:.3f}")
ch = (sl[sl["period"] != "2024 (transition)"].groupby(["store_id", "format", "remodeled_2024", "period"])[["num", "den"]]
        .apply(lambda g: g["num"].sum() / g["den"].sum()).unstack("period"))
ch["change"] = np.log(ch["2025 (after)"] / ch["2023 (before)"])
print("\nSame store, 2023 -> 2025 change in log-lift (difference in differences):")
print(ch.groupby(["format", "remodeled_2024"])["change"].agg(["size", "mean"]).round(3).to_string())
a = ch.loc[ch.index.get_level_values("remodeled_2024"), "change"]; bb = ch.loc[~ch.index.get_level_values("remodeled_2024"), "change"]
u, p = stats.mannwhitneyu(a, bb, alternative="two-sided")
print(f"  remodeled n={len(a)} mean {a.mean():+.3f} | others n={len(bb)} mean {bb.mean():+.3f} | Mann-Whitney p={p:.3f}")


Belief 1 - lift and extra units per store-day, by speed quartile within format:
                  n  base  mean_lift  median_lift  extra_units
format  speed                                                 
Express Q1 slow  37  0.18       2.04         1.87         0.17
        Q2       36  0.38       1.88         1.91         0.33
        Q3       36  0.59       1.67         1.58         0.39
        Q4 fast  37  1.19       1.42         1.41         0.48
Hyper   Q1 slow  63  0.71       3.21         2.96         1.47
        Q2       62  1.50       3.08         2.95         3.08
        Q3       62  2.56       2.99         2.95         5.09
        Q4 fast  63  5.26       2.97         2.82        10.20
Super   Q1 slow  63  0.36       2.30         2.21         0.46
        Q2       62  0.73       2.18         2.15         0.86
        Q3       62  1.29       2.27         2.26         1.65
        Q4 fast  63  2.49       2.12         2.09         2.76

Spearman correlation log(baseline) ~ 


Belief 2 - store x promotion lift by remodel status (sum / sum, within format):
                                          n_store_promos  lift
period            format  remodeled_2024                      
2023 (before)     Express False                        4  1.43
                          True                         4  1.36
                  Hyper   False                       16  2.83
                          True                        16  3.37
                  Super   False                       20  1.95
                          True                        36  2.29
2024 (transition) Express False                       28  1.63
                          True                        28  1.49
                  Hyper   False                       40  2.64
                          True                        40  3.16
                  Super   False                       50  1.95
                          True                        90  2.25
2025 (after)      Express False      

## April 2025: supply, not demand (C1; ruled-out explanations a, b)
Weekly opening stock, sales and empty-shelf rate, 2025 vs 2024; the four April weeks by category.

In [27]:
"""Ruled-out check: was the April 2025 drop a demand drop or a supply problem?
Weekly opening stock, sales and empty-shelf rate for all store-items, spring 2025 vs the same weeks in 2024.
Run: python check_april_supply.py <data_folder> <output_folder>"""
import pandas as pd

df = F.df
def weekly(a, b):
    w = df[(df["date"] >= a) & (df["date"] <= b)].copy()
    w["week"] = w["date"].dt.to_period("W-SUN").dt.start_time.dt.strftime("%d %b %Y")
    t = w.groupby("week", sort=False).agg(sales_per_store_item_day=("units_sold", "mean"),
                                           opening_stock=("stock_on_hand_open", "mean"),
                                           empty_shelf_rate=("shelf_empty", "mean"),
                                           zero_stock_rate=("stock_on_hand_open", lambda s: (s == 0).mean()))
    return t.round(3)
t25 = weekly("2025-03-03", "2025-06-15"); t24 = weekly("2024-03-04", "2024-06-16")
print("2025:\n", t25.to_string()); print("\n2024, same weeks:\n", t24.to_string())
# which categories were hit hardest in the four April weeks
w = df[(df["date"] >= "2025-04-07") & (df["date"] <= "2025-05-04")]
ref = df[(df["date"] >= "2025-03-10") & (df["date"] <= "2025-04-06")]
c = pd.DataFrame({"april_stock": w.groupby("category_id")["stock_on_hand_open"].mean(),
                  "march_stock": ref.groupby("category_id")["stock_on_hand_open"].mean(),
                  "april_empty_rate": w.groupby("category_id")["shelf_empty"].mean(),
                  "march_empty_rate": ref.groupby("category_id")["shelf_empty"].mean()})
c["stock_change"] = (c["april_stock"] / c["march_stock"] - 1).round(3)
print("\nBy category, 7 Apr-4 May vs 10 Mar-6 Apr:\n", c.round(3).sort_values("stock_change").to_string())
t25.to_csv(os.path.join(F.OUT, "april_weekly_2025.csv")); t24.to_csv(os.path.join(F.OUT, "april_weekly_2024.csv"))
c.round(3).to_csv(os.path.join(F.OUT, "april_by_category.csv"))


2025:
              sales_per_store_item_day  opening_stock  empty_shelf_rate  zero_stock_rate
week                                                                                   
03 Mar 2025                     1.656          8.812             0.130            0.077
10 Mar 2025                     1.482          8.618             0.110            0.065
17 Mar 2025                     1.517          8.546             0.112            0.064
24 Mar 2025                     1.538          8.463             0.116            0.066
31 Mar 2025                     1.529          8.486             0.114            0.066
07 Apr 2025                     1.458          7.613             0.161            0.096
14 Apr 2025                     1.515          7.662             0.182            0.114
21 Apr 2025                     1.419          7.462             0.184            0.117
28 Apr 2025                     1.428          7.437             0.183            0.117
05 May 2025              

## Empty-shelf days: exclude or fill? (C7; ruled-out explanation c)
Empty-shelf rate by weekday and format; the lift computed three ways on the same events.

In [28]:
"""Ruled-out check: does excluding empty-shelf days bias the lift because those are the high-demand days?
(1) Empty-shelf rate by weekday and by format, normal vs promotion days.
(2) On promotion days: full-shelf mean sales vs the opening stock (lower bound) of the days that emptied.
(3) The lift computed three ways on the same item x promotion x format events: all days (raw), empty days
    excluded, empty days filled from the same promotion's full-shelf store-days (demand rule).
Run: python test_empty_shelf_treatment.py <data_folder> <output_folder>"""
import numpy as np, pandas as pd

df, promo = F.df, F.promo
days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
t = df.groupby(["promo", "wd"])["shelf_empty"].mean().unstack("wd"); t.columns = days; t.index = ["normal day", "promotion day"]
print("Empty-shelf rate by weekday:\n", t.round(3).to_string())
prof = df[~df["promo"] & ~df["shelf_empty"]].groupby("wd")["units_sold"].mean(); prof.index = days
print("\nNormal days, full shelf, sales profile (mean = 1):\n", (prof / prof.mean()).round(2).to_string())
t2 = df.groupby(["promo", "format"])["shelf_empty"].mean().unstack("format"); t2.index = ["normal day", "promotion day"]
print("\nEmpty-shelf rate by format:\n", t2.round(3).to_string())
pg = df[df["promo"]]
print(f"\nPromotion days: full-shelf mean sales = {pg.loc[~pg['shelf_empty'], 'units_sold'].mean():.2f}; "
      f"emptied days' opening stock (lower bound) = {pg.loc[pg['shelf_empty'], 'stock_on_hand_open'].mean():.2f}")

def fill_rule(g):
    """empty day -> mean of the group's full-shelf days with sales >= that day's opening stock; else the stock itself"""
    full = g.loc[~g["shelf_empty"], "units_sold"].to_numpy()
    out = g["units_sold"].to_numpy().astype(float)
    for i, (e, s) in enumerate(zip(g["shelf_empty"].to_numpy(), g["stock_on_hand_open"].to_numpy())):
        if e:
            u = full[full >= s]; out[i] = u.mean() if len(u) else s
    return out

rows = []
for p in promo.itertuples():
    for fmt in [x.strip() for x in p.formats.split(",")]:
        m = (df["item_id"] == p.item_id) & (df["format"] == fmt)
        pre = df[m & df["date"].between(p.start_date - pd.Timedelta(days=21), p.start_date - pd.Timedelta(days=1))]
        week = df[m & df["date"].between(p.start_date, p.end_date)]
        if len(week) == 0 or len(pre) == 0:
            continue
        pre_f = np.concatenate([fill_rule(g) for _, g in pre.groupby("store_id")])   # before: per store-item
        week_f = fill_rule(week)                                                     # promotion week: per item-promotion-format
        pre_x, week_x = pre.loc[~pre["shelf_empty"], "units_sold"], week.loc[~week["shelf_empty"], "units_sold"]
        rows.append(dict(promo_id=p.promo_id, item_id=p.item_id, format=fmt, discount=p.discount,
                         empty_rate_week=week["shelf_empty"].mean(),
                         raw=week["units_sold"].mean() / pre["units_sold"].mean() if pre["units_sold"].mean() > 0 else np.nan,
                         excluded=week_x.mean() / pre_x.mean() if len(week_x) and len(pre_x) and pre_x.mean() > 0 else np.nan,
                         filled=week_f.mean() / pre_f.mean() if pre_f.mean() > 0 else np.nan))
lc = pd.DataFrame(rows).replace([np.inf, -np.inf], np.nan).dropna()
print(f"\nLift three ways, item x promotion x format, n = {len(lc)}")
print(lc.groupby("format")[["raw", "excluded", "filled"]].mean().round(2).to_string())
lc["empty_group"] = pd.cut(lc["empty_rate_week"], [0, 0.1, 0.25, 0.5, 1.0], include_lowest=True)
print("\nBy the share of empty-shelf store-days in the promotion week:")
print(lc.groupby("empty_group", observed=True)[["raw", "excluded", "filled"]].agg(["mean", "count"]).round(2).to_string())
r = lc["filled"] / lc["excluded"]
print(f"\nfilled / excluded per event: median {r.median():.2f}; events > +10 %: {(r > 1.1).sum()}, < -10 %: {(r < 0.9).sum()}")
lc.to_csv(os.path.join(F.OUT, "lift_three_ways.csv"), index=False)


Empty-shelf rate by weekday:
                  Mon    Tue    Wed    Thu    Fri    Sat    Sun
normal day     0.193  0.135  0.108  0.058  0.092  0.165  0.256
promotion day  0.228  0.240  0.237  0.050  0.150  0.329  0.474



Normal days, full shelf, sales profile (mean = 1):
 Mon    0.90
Tue    0.89
Wed    0.93
Thu    0.99
Fri    1.14
Sat    1.24
Sun    0.91



Empty-shelf rate by format:
 format         Express  Hyper  Super
normal day       0.252  0.112  0.125
promotion day    0.362  0.293  0.189

Promotion days: full-shelf mean sales = 3.90; emptied days' opening stock (lower bound) = 2.57



Lift three ways, item x promotion x format, n = 647
          raw  excluded  filled
format                         
Express  1.70      1.99    1.89
Hyper    2.54      2.94    3.07
Super    2.12      2.25    2.30

By the share of empty-shelf store-days in the promotion week:
                raw       excluded       filled      
               mean count     mean count   mean count
empty_group                                          
(-0.001, 0.1]  1.89    52     1.99    52   1.85    52
(0.1, 0.25]    2.17   330     2.32   330   2.34   330
(0.25, 0.5]    2.41   226     2.84   226   2.97   226
(0.5, 1.0]     1.44    39     2.03    39   2.00    39

filled / excluded per event: median 1.02; events > +10 %: 93, < -10 %: 53
